In [1]:
%pip --version

pip 25.0.1 from d:\hanwha_0902\hanwha_0902\ex0916\.venv\Lib\site-packages\pip (python 3.12)

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -qU langchain-teddynote

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
%pip install --upgrade pip

  Using cached pip-26.2.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.2.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.0.1
    Uninstalling pip-25.0.1:
      Successfully uninstalled pip-25.0.1
Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
#7 Pandas 활용
import pprint
from typing import Any, Dict

import pandas as pd
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("test_0916")

class PandasQuery(BaseModel) :
    column: str = Field(description="분석할 컬럼 이름")
    operation: str = Field(
        description="수행할 연산. 예: mean, count, sum, min, max, quantile"
    )
    filter_column: str | None = Field(
        default=None,
        description="필터링에 사용할 컬럼"
    )
    filter_value: int | float | str | None = Field(
        default=None,
        description="필터링할 값"
    )

model = ChatOpenAI(temperature=0.5, model_name="gpt-4o-mini")
structured_model = model.with_structured_output(PandasQuery)

def format_parser_output(parser_output: Dict[str, Any]) -> None:
    for key in parser_output.keys():
        parser_output[key] = parser_output[key].to_dict()
    return pprint.PrettyPrinter(width=40, compact=True).pprint(parser_output)

df = pd.read_csv("./titanic.csv")
df.head(10)

LangSmith 추적을 시작합니다.
[프로젝트명]
test_0916


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [4]:
prompt = f"""
다음 DataFrame의 컬럼은 아래와 같습니다.

{df.columns.tolist()}

사용자의 질문을 분석해서 PandasQuery 형식으로 반환하세요.
"""
pprint.pprint(df.columns.tolist())

['PassengerId',
 'Survived',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Cabin',
 'Embarked']


In [12]:
from langchain_core.prompts import PromptTemplate

df_query = "Age column을 조회해 주세요."

prompt = PromptTemplate(
    template="""
Answer the user query.

{format_instructions}

IMPORTANT:
Return ONLY the format specified above.
Do NOT return JSON.
Do NOT use quotation marks.
Do NOT add explanations.

Question:
{question}
""",
    input_variables=["question"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    },
)

chain = prompt | model | parser

parser_output = chain.invoke({"question": df_query})

pprint.pprint(parser_output)

{'Age': 0     22.0
1     38.0
2     26.0
3     35.0
4     35.0
5      NaN
6     54.0
7      2.0
8     27.0
9     14.0
10     4.0
11    58.0
12    20.0
13    39.0
14    14.0
15    55.0
16     2.0
17     NaN
18    31.0
19     NaN
Name: Age, dtype: float64}


In [13]:
df_query = "Retrieve the first row."
parser_output= chain.invoke({"question": df_query})
pprint.pprint(parser_output)
df["Age"].head().mean()

{'0': PassengerId                          1
Survived                             0
Pclass                               3
Name           Braund, Mr. Owen Harris
Sex                               male
Age                               22.0
SibSp                                1
Parch                                0
Ticket                       A/5 21171
Fare                              7.25
Cabin                              NaN
Embarked                             S
Name: 0, dtype: object}


np.float64(31.2)

In [24]:
df_query="Retrieve the quantile of the Ages from row 0 to 15."
parser_output=chain.invoke({"question":df_query})
print(parser_output)

{'quantile': np.float64(27.0)}


In [25]:
#8 Datetime 출력 파서
from pydantic import BaseModel
from datetime import datetime
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("test_0916")


class Meeting(BaseModel):
    meeting_time: datetime

llm=ChatOpenAI(model="gpt-4o")
structured_llm = llm.with_structured_output(Meeting)


result = structured_llm.invoke(
    "회의 일정은 2026년 9월 18일 오전 11시입니다."
)

print(result)
print(result.meeting_time)

LangSmith 추적을 시작합니다.
[프로젝트명]
test_0916
meeting_time=datetime.datetime(2026, 9, 18, 11, 0, tzinfo=TzInfo(0))
2026-09-18 11:00:00+00:00


In [ ]:
%pip install langchain_classic

In [28]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_classic.output_parsers import DatetimeOutputParser
from langchain_classic.prompts import PromptTemplate
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("test_0916")

llm = ChatOpenAI(model="gpt-4o")

output_parser = DatetimeOutputParser()

output_parser.format = "%Y-%m-%d"

print(output_parser.get_format_instructions())

LangSmith 추적을 시작합니다.
[프로젝트명]
test_0916
Write a datetime string that matches the following pattern: '%Y-%m-%d'.

Examples: 2026-09-16, 2025-09-16, 2026-09-15

Return ONLY this string, no other words!


In [30]:
template = """Answer the users question:

#Format Instructions:
# {format_instructions}
# Question: {question}
# 
# Answer: """

prompt = PromptTemplate.from_template(
    template,
    partial_variables={
        "format_instructions": output_parser.get_format_instructions()
    },
)

prompt

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={'format_instructions': "Write a datetime string that matches the following pattern: '%Y-%m-%d'.\n\nExamples: 2026-09-16, 2025-09-16, 2026-09-15\n\nReturn ONLY this string, no other words!"}, template='Answer the users question:\n\n#Format Instructions:\n# {format_instructions}\n# Question: {question}\n# \n# Answer: ')

In [ ]:
chain = prompt | ChatOpenAI() | output_parser
output = chain.invoke({"question": "이치란 라멘이 창업한 연도"})

output.strftime("%Y-%m-%d")

'2020-09-16'

In [1]:
#Enum 열거형 출력 파서
from enum import Enum
from langchain_classic.output_parsers.enum import EnumOutputParser
from langchain_classic.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("test_0916")

class Colors(Enum):
    RED = "빨간색"
    GREEN = "초록색"
    BLUE = "파란색"

Colors.RED

LangSmith 추적을 시작합니다.
[프로젝트명]
test_0916


<Colors.RED: '빨간색'>

In [3]:
parser = EnumOutputParser(enum=Colors)
parser.get_format_instructions()

'Select one of the following options: 빨간색, 초록색, 파란색'

In [11]:
prompt = PromptTemplate.from_template(
    """다음의 물체는 어떤 색깔인가요?
    
Object : {object}

Instructions : {instructions}"""
).partial(instructions=parser.get_format_instructions())

chain = prompt | ChatOpenAI() | parser

response = chain.invoke({"object" : "손흥민"})
print(response)

Colors.RED


In [12]:
type(response)

<enum 'Colors'>

In [13]:
response.value

'빨간색'